# Phase 2 — Sentiment Analysis with FinBERT

## What is FinBERT?
FinBERT is a BERT model fine-tuned specifically on financial text —
earnings calls, analyst reports, SEC filings. Unlike general sentiment
tools like VADER, FinBERT understands financial language nuances.

## Why not VADER?
VADER was trained on social media. It doesn't understand:
- "Revenue headwinds" = negative
- "Margin compression" = negative  
- "Significant runway ahead" = positive

FinBERT was trained on 4.9 billion tokens of financial text.

## Approach
1. Load each company's earnings text
2. Split into 512-token chunks (FinBERT limit)
3. Run sentiment on each chunk
4. Aggregate to company-level score
5. Save results for Phase 3 correlation analysis

In [2]:
import pandas as pd
import os
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np

# load FinBERT model
# this downloads ~440MB on first run - normal
print("Loading FinBERT model...")
print("(First run downloads ~440MB - this is normal)")

tokenizer = BertTokenizer.from_pretrained('ProsusAI/finbert')
model = BertForSequenceClassification.from_pretrained('ProsusAI/finbert')
model.eval()  # set to evaluation mode, not training

print("FinBERT loaded successfully")
print(f"Labels: {model.config.id2label}")

Loading FinBERT model...
(First run downloads ~440MB - this is normal)
FinBERT loaded successfully
Labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [3]:
def analyze_sentiment(text, chunk_size=512):
    """
    Run FinBERT sentiment on a long text by chunking it
    Returns average positive, negative, neutral scores
    """
    # split text into words first
    words = text.split()
    
    # create chunks of roughly chunk_size tokens
    # using words as proxy for tokens (close enough)
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    
    all_scores = []
    
    for chunk in chunks:
        # tokenize the chunk
        inputs = tokenizer(
            chunk,
            return_tensors='pt',
            truncation=True,
            max_length=512,
            padding=True
        )
        
        # run through model - no gradient needed for inference
        with torch.no_grad():
            outputs = model(**inputs)
        
        # convert logits to probabilities
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        scores = probs[0].tolist()
        
        # scores order: positive, negative, neutral
        all_scores.append({
            'positive': scores[0],
            'negative': scores[1],
            'neutral': scores[2]
        })
    
    # average across all chunks
    avg_positive = np.mean([s['positive'] for s in all_scores])
    avg_negative = np.mean([s['negative'] for s in all_scores])
    avg_neutral = np.mean([s['neutral'] for s in all_scores])
    
    # overall sentiment label
    if avg_positive > avg_negative and avg_positive > avg_neutral:
        label = 'Positive'
    elif avg_negative > avg_positive and avg_negative > avg_neutral:
        label = 'Negative'
    else:
        label = 'Neutral'
    
    return {
        'positive': round(avg_positive, 4),
        'negative': round(avg_negative, 4),
        'neutral': round(avg_neutral, 4),
        'label': label,
        'chunks_analyzed': len(chunks)
    }

# test with Apple text
print("Testing on Apple earnings text...")
apple_text = open('data/texts/AAPL_earnings.txt').read()
result = analyze_sentiment(apple_text)
print(f"Result: {result}")

Testing on Apple earnings text...
Result: {'positive': np.float64(0.2519), 'negative': np.float64(0.0387), 'neutral': np.float64(0.7094), 'label': 'Neutral', 'chunks_analyzed': 4}


In [4]:
results = []

for ticker in ['AAPL', 'MSFT', 'JPM', 'JNJ', 'AMZN', 'GOOGL', 'BAC', 'PFE', 'V', 'UNH']:
    filepath = f'data/texts/{ticker}_earnings.txt'
    
    if not os.path.exists(filepath):
        print(f"{ticker}: file not found")
        continue
    
    print(f"Analyzing {ticker}...")
    text = open(filepath).read()
    sentiment = analyze_sentiment(text)
    
    results.append({
        'ticker': ticker,
        'positive': sentiment['positive'],
        'negative': sentiment['negative'],
        'neutral': sentiment['neutral'],
        'label': sentiment['label'],
        'chunks': sentiment['chunks_analyzed']
    })
    
    print(f"  {sentiment['label']} - pos:{sentiment['positive']:.3f} neg:{sentiment['negative']:.3f}")

df_sentiment = pd.DataFrame(results)
print(f"\nSentiment Analysis Complete")
print(df_sentiment[['ticker', 'label', 'positive', 'negative', 'neutral']])

Analyzing AAPL...
  Neutral - pos:0.252 neg:0.039
Analyzing MSFT...
  Neutral - pos:0.173 neg:0.228
Analyzing JPM...
  Neutral - pos:0.390 neg:0.143
Analyzing JNJ...
  Neutral - pos:0.348 neg:0.072
Analyzing AMZN...
  Neutral - pos:0.249 neg:0.113
Analyzing GOOGL...
  Neutral - pos:0.240 neg:0.161
Analyzing BAC...
  Neutral - pos:0.370 neg:0.094
Analyzing PFE...
  Neutral - pos:0.193 neg:0.192
Analyzing V...
  Neutral - pos:0.241 neg:0.089
Analyzing UNH...
  Neutral - pos:0.088 neg:0.334

Sentiment Analysis Complete
  ticker    label  positive  negative  neutral
0   AAPL  Neutral    0.2519    0.0387   0.7094
1   MSFT  Neutral    0.1727    0.2277   0.5996
2    JPM  Neutral    0.3901    0.1434   0.4666
3    JNJ  Neutral    0.3478    0.0724   0.5798
4   AMZN  Neutral    0.2488    0.1129   0.6383
5  GOOGL  Neutral    0.2400    0.1611   0.5988
6    BAC  Neutral    0.3697    0.0937   0.5366
7    PFE  Neutral    0.1927    0.1923   0.6150
8      V  Neutral    0.2413    0.0895   0.6692
9    UNH